In [1]:
from netgen.occ import *
import ngsolve
from ngsolve import *
from ngsolve.webgui import Draw

In [2]:
box = Box((0,0,0), (3,0.6,1))
box.faces.name="outer"
cyl = sum( [Cylinder((0.5+i,0,0.5), Y, 0.25,0.8) for i in range(3)] )
cyl.faces.name="cyl"
geo = box-cyl


Draw(geo, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3,…

BaseWebGuiScene

In [3]:
cylboxedges = geo.faces["outer"].edges * geo.faces["cyl"].edges
cylboxedges.name = "cylbox"
geo = geo.MakeChamfer(cylboxedges, 0.03)

In [4]:
geo.faces.Min(X).name = "fix"
geo.faces.Max(X).name = "force"

Draw(geo, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3,…

In [5]:
from ngsolve import *
from ngsolve.webgui import Draw
mesh = Mesh(OCCGeometry(geo).GenerateMesh(maxh=0.1)).Curve(3)
Draw(mesh, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

In [6]:
dt = 0.1
E, nu = 210, 0.2
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

def Stress(strain):
    return 2*mu*strain + lam*Trace(strain)*Id(3)

In [7]:
t = Parameter(0.0)
fes = VectorH1(mesh, order=3, dirichlet="fix")
u,v = fes.TnT()
gfu = GridFunction(fes)

with TaskManager():
    a = BilinearForm(InnerProduct(Stress(Sym(Grad(u))), Sym(Grad(v))).Compile()*dx)
    pre = Preconditioner(a, "bddc")
    a.Assemble()
    m = BilinearForm(u*v*dx)
    pre = Preconditioner(a, "bddc")
    m.Assemble()

In [8]:
force = CF( (5,0,0))*cos(2*3.1415*t) 
f = LinearForm(force*v*ds("force"))

In [9]:
from ngsolve.krylovspace import CGSolver
mstar = m.mat.CreateMatrix()
mstar.AsVector().data = m.mat.AsVector() + dt * a.mat.AsVector()
inv = CGSolver(mstar, pre, printrates=True, tol=1e-8)

: 

In [ ]:
gfut = GridFunction(gfu.space,multidim=0)
tend = t.Get()+0.5
while t.Get() <= tend:
    print(f"Time: {t.Get()}")
    f.Assemble()

    rhs = dt * f.vec + m.mat * gfu.vec
    gfu.vec.data = inv * rhs
    gfut.AddMultiDimComponent(gfu.vec)
    
    new_t = t.Get()+dt
    t.Set(new_t)
    
    

In [11]:
with TaskManager():
    fesstress = MatrixValued(H1(mesh,order=3), symmetric=True)
    gfstress = GridFunction(fesstress)
    gfstress.Interpolate (Stress(Sym(Grad(gfu))))

In [12]:
Draw (gfut, mesh, height="2vh", interpolate_multidim=True, animate=True)

WebGuiWidget(layout=Layout(height='2vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

BaseWebGuiScene

In [13]:
Draw (Norm(gfstress), mesh, deformation=1e4*gfu, draw_vol=False, order=3, height="2vh");

WebGuiWidget(layout=Layout(height='2vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…